In [11]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
import json

import load_data as ld
import candidates as ds
import priority_queue as pq
import subgroups as s
import refinements as rf
import subgroup_selection as ss
import search as bs
import preprocessing as pp
import candidates as ca
import quality_measure as qm

In [2]:
# UNCOMMENT If you run for the first time without the CSV ready -> gets firstbatch_2

# pp.transform_jsons()

In [3]:
df = pd.read_csv('algebrakit_firstbatch_2.csv')
collection_attributes = ['skills', 'stepTypes', 'event_types', 'event_evaluations', 'positive_skills', 'negative_skills']

for colatt in collection_attributes:
    df[colatt] = df[colatt].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df.head(2)

,exerciseID,sessionID,success,creationTimestamp,skills,stepTypes,finished,marksEarned,marksTotal,questionID,...,result_progress,marksEarned_interaction,marksTotal_interaction,finished_interaction,maxAttempts,usedAttempts,failed,marksPenalty,hintsRequested,mathErrors
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q1,...,1.0,1.0,1.0,True,2.0,0.0,False,0.0,0,0
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.0,4.0,Q2,...,0.0,0.0,1.0,False,2.0,2.0,True,0.0,1,2


In [4]:
df['interactionType'].unique()

array(['MULTISTEP', 'FILL_IN_THE_BLANKS'], dtype=object)

# Data distribution in the Use Case
Our data can be split into two categories: the **exercise attributes** that describe the contents of an exercise and the **student attributes** that describe the workflow and performance of the students that made the exercise. The hierarchical level that each attribute is at, is denoted behind the attribute name.

**Exercise Attributes:**
* skills (eid)
* stepTypes (eid)
* marksTotal (eid)
* itemType (qid)
* interactionType (iid)
* solution (iid)
* content (iid)
* interaction marksTotal (iid)
* maxAttempts (iid)

**Student Attributes:**
* success (sid)
* finished (sid)
* marksEarned (sid)
* eventTypes (iid)
* eventEvaluations (iid)
* positive skills (iid)
* negative skills (iid)
* result status (iid)
* result progress (iid)
* interaction marksEarned (iid)
* interaction finished (iid)
* usedAttempts (iid)
* failed (iid)
* marksPenalty (iid)
* hintsRequested (iid)
* mathErrors (iid)

## Attribute Types and Descriptive and Target Space.
That we have this many attributes, does not mean that we select all attributes. We therefore select only a part of it. These attributes can be composed of six categories: numerical (float), ordinal (int), categorical, boolean, collection, sequence, and subsequence. The last three are list-type attributes that simulate the aggregation of a hierarchical level that has no need to be expanded into seperate rows in the data. 

**Target Attribute**
* finished (boolean)

The below mentioned are our descriptors:

**Numerical (float)**
* marksPenalty

**Ordinal (int)**
* marksTotal
* interaction marksTotal
* usedAttempts
* marksPenalty
* hintsRequested
* mathErrors

**Categorical**
* itemType
* interactionType

**Boolean**
* nvt

**Collection (list, no order)**
* skills
* positive skills
* negative skills

**Sequence (list, order immediate)**
* eventTypes
* eventEvaluations

**Subsequence (list, order with pauses)**
* stepTypes (also possible as collection or sequence)

In [5]:
df.columns

Index(['exerciseID', 'sessionID', 'success', 'creationTimestamp', 'skills',
       'stepTypes', 'finished', 'marksEarned', 'marksTotal', 'questionID',
       'questionType', 'interactionID', 'itemType', 'interactionType',
       'solution', 'content', 'event_types', 'event_evaluations',
       'positive_skills', 'negative_skills', 'result_status',
       'result_progress', 'marksEarned_interaction', 'marksTotal_interaction',
       'finished_interaction', 'maxAttempts', 'usedAttempts', 'failed',
       'marksPenalty', 'hintsRequested', 'mathErrors'],
      dtype='object')

In [6]:
descriptors = [ 'itemType', 'interactionType', #categorical
               'skills', 'positive_skills', 'negative_skills', #collection 
               'event_types', 'event_evaluations',  #sequence
               'stepTypes', #subsequence
               'marksTotal', 'marksTotal_interaction', 'usedAttempts', 'marksPenalty', 'hintsRequested', 'mathErrors'] #ordinal


targets = ['finished'] # all to do with performance

ids = ['exerciseID', 'sessionID', 'questionID', 'interactionID']

In [21]:
# +- 12 mins
refinement_params = {'numerical_bins': 3, 'collection_limit': 3}
beam_params = {'k': 10, 'd': 3, 'w': 20}
constraint_params = {'minimum_size': 25}
model_params = {'model': 'F_beta', 'beta': 1, 'weighted': True}

result_set = bs.beam_search(data=df, descriptors=descriptors, targets=targets, ids=ids, 
                            refinement_params=refinement_params, beam_params=beam_params,
                            constraint_params=constraint_params, model_params=model_params)

start on level 1
Last beam was The priority queue has 20 subgroups.
start on level 2
Last beam was The priority queue has 20 subgroups.
start on level 3
Last beam was The priority queue has 20 subgroups.


In [22]:
result_set.summarize_queue()

This queue has subgroups of average size 5.0 in a queue of length 10 and an average quality of 0.5990265939378769.


In [23]:
for sub in result_set._queue:
    print(sub.__str__())
    print(sub._metrics['session_counts'])
    print(56 * "=========")

This subgroup of size 3 has description {'skills': ('FractionArithmeticSimplifyProper', 'AlgebraAddingLikeTerms')} and a quality score of 0.6470552345290468.
[123, 84, 61]
This subgroup of size 14 has description {'interactionType': (1.0, 'MULTISTEP')} and a quality score of 0.6220539406006752.
[123, 110, 124, 91, 97, 84, 167, 127, 86, 129, 169, 105, 114, 96]
This subgroup of size 2 has description {'skills': ('EquationsBalanceMethodArrangeTerms', 'AlgebraAddingLikeTerms'), 'interactionType': (0.0, 'MULTISTEP')} and a quality score of 0.6137299052554145.
[110, 169]
This subgroup of size 2 has description {'skills': ('AlgebraAddingLikeTerms',), 'interactionType': (1.0, 'FILL_IN_THE_BLANKS')} and a quality score of 0.6137299052554145.
[110, 169]
This subgroup of size 4 has description {'skills': ('EquationsBalanceMethodArrangeTerms', 'AlgebraAddingLikeTerms')} and a quality score of 0.6116725752333194.
[110, 84, 96, 169]
This subgroup of size 4 has description {'interactionType': (1.0, '

In [29]:
ex7_idx = result_set._queue[9]._indices

df_ex7 = df.iloc[ex7_idx]
list( df_ex7['exerciseID'].unique() )

['ba15ab94-5da4-414e-a01d-107ea7c34d67',
 '58b0759d-48ef-49f3-bb5f-7d999f35b12b',
 '086f5a82-c080-4d05-9431-03e992a0d7b3']

In [25]:
qm.finished_rates(df[ids + targets])

({'086f5a82-c080-4d05-9431-03e992a0d7b3': {False: 118, True: 5},
  '0beaa764-f558-4124-8d4d-b224ebdb2d4a': {False: 94, True: 16},
  '264d9ab3-57ba-4861-855b-fcc2a465c5db': {False: 114, True: 10},
  '315bc927-cdd9-4d2a-8b3c-68f830f74e70': {False: 48, True: 43},
  '486d7a7a-f282-48f4-8b74-c01a1c23a971': {False: 69, True: 28},
  '58b0759d-48ef-49f3-bb5f-7d999f35b12b': {False: 57, True: 27},
  '9aec3c1a-0100-40dd-a146-bdee6ff72fb3': {False: 127, True: 40},
  '9f4a675e-c2b3-4e11-91cc-4445951bab60': {False: 96, True: 31},
  'a04508aa-09fa-4782-bfc3-effc3789fd47': {False: 35, True: 51},
  'ba15ab94-5da4-414e-a01d-107ea7c34d67': {False: 113, True: 16},
  'bb996be4-f9a9-48fb-9a0c-61e52a67fd69': {False: 100, True: 69},
  'd4399f1b-009a-49c2-8d5f-b1ac805d195f': {False: 81, True: 24},
  'd93986a5-e93a-4be1-98cc-4d6ec991ef4f': {False: 81, True: 33},
  'e3faf215-08e9-4ec0-b960-a35ccf261c7d': {False: 64, True: 32}},
 [123, 110, 124, 91, 97, 84, 167, 127, 86, 129, 169, 105, 114, 96])